# Paper numbers validation 

Validates the key statistics from the paper description against `all_obs.csv` (7.7M cells).

In [27]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

CSV = "/workspace/perturbai-private/notebooks/reviews/data/Figure_1/all_obs.csv"
obs = pd.read_csv(CSV, index_col=0, low_memory=False)

# Normalize safe_target variants (e.g. Safe_target001, safe_targetAAV) → 'Safe_target'
_n_before = obs["gene_target"].nunique()
obs["gene_target"] = obs["gene_target"].str.replace(
    r'(?i)^safe_target\w*', 'Safe_target', regex=True
)
_n_after = obs["gene_target"].nunique()
print(f"gene_target unique values: {_n_before:,} → {_n_after:,} after collapsing safe_target variants")

# --- QC filter ---
singlet   = obs["scDblFinder.class"] == "singlet"
min_genes = obs["num_genes"] >= 2000
ambient   = obs["log_ambient_mse_norm"] > 0.09
pp_filter = singlet & min_genes & ambient

obs_filt = obs[pp_filter].copy()

# ── 1. cell counts ──────────────────────────────────────────────────────────
total    = len(obs)
pp_cells = len(obs_filt)

# ── 2. taxonomy ─────────────────────────────────────────────────────────────
n_classes    = obs["predicted_class"].nunique()
n_subclasses = obs["predicted_subclass"].nunique()

# ── 3. neighborhoods ────────────────────────────────────────────────────────
core_hoods    = ["Pallium-Glut", "Subpallium-GABA", "HY-EA-Glut-GABA",
                 "TH-EPI-Glut", "MB-HB-Glut-Sero-Dopa", "MB-HB-CB-GABA", "NN-IMN-GC"]
neuronal_core = [h for h in core_hoods if h != "NN-IMN-GC"]

# ── 4. UMI medians ──────────────────────────────────────────────────────────
def umi_range(df):
    medians = {h: df[df["neighborhood"] == h]["num_rna_umi"].median() for h in neuronal_core}
    medians = {h: v for h, v in medians.items() if not np.isnan(v)}
    return int(min(medians.values())), int(max(medians.values()))

umi_min_7m,   umi_max_7m   = umi_range(obs)
umi_min_filt, umi_max_filt = umi_range(obs_filt)

# ── 5. guide detection ──────────────────────────────────────────────────────
def guide_pcts(df):
    n = len(df)
    return (df["num_guides"] >= 1).sum() / n * 100, (df["num_guides"] == 1).sum() / n * 100

guide_all_pct_7m,   single_all_pct_7m   = guide_pcts(obs)
guide_all_pct_filt, single_all_pct_filt = guide_pcts(obs_filt)

# ── 6. perturbation coverage — with and without single-guide filter ──────────
def pert_stats(df, single_guide_only):
    base = df[df["num_guides"] == 1] if single_guide_only else df
    counts         = base["gene_target"].value_counts()
    n_targets      = len(counts)
    median_cells   = int(counts.median())
    sub_pairs_gt50 = int((base.groupby(["gene_target", "predicted_subclass"]).size() >= 50).sum())
    grp_pairs_gt50 = int((base.groupby(["gene_target", "predicted_group"]).size()  >= 50).sum())
    return n_targets, median_cells, sub_pairs_gt50, grp_pairs_gt50

n_tgt_7m_sg,  med_7m_sg,  sub_7m_sg,  grp_7m_sg  = pert_stats(obs,      True)
n_tgt_7m_all, med_7m_all, sub_7m_all, grp_7m_all = pert_stats(obs,      False)
n_tgt_fi_sg,  med_fi_sg,  sub_fi_sg,  grp_fi_sg  = pert_stats(obs_filt, True)
n_tgt_fi_all, med_fi_all, sub_fi_all, grp_fi_all = pert_stats(obs_filt, False)

gene_target unique values: 1,016,832 → 1,008,324 after collapsing safe_target variants


In [28]:
from IPython.display import display, HTML

# ── Table 1: full 7.7M obs ───────────────────────────────────────────────────
rows_7m = [
    ("Total nuclei",
     f"{total:,}",
     "7,720,300"),
    ("Cell classes (predicted_class)",
     str(n_classes),
     "34"),
    ("Cell subclasses (predicted_subclass)",
     str(n_subclasses),
     "331"),
    ("Neighborhoods",
     "7 core (+4 compound overlap labels)",
     "7"),
    ("Median UMI/nucleus — range across neuronal neighborhoods",
     f"{umi_min_7m:,} – {umi_max_7m:,}",
     "11,404 – 14,652"),
    ("% nuclei with guide detected",
     f"{guide_all_pct_7m:.2f}%",
     ">66.49%"),
    ("% nuclei with single guide",
     f"{single_all_pct_7m:.2f}%",
     "50%"),
    ("— Perturbation stats: single-guide cells only —", "", ""),
    ("  Unique gene targets",
     f"{n_tgt_7m_sg:,}",
     "~2,000"),
    ("  Median nuclei per perturbation",
     f"{med_7m_sg:,}",
     "1,795"),
    ("  Pert × subclass pairs >50 cells",
     f"{sub_7m_sg:,}",
     "—"),
    ("  Pert × group pairs >50 cells",
     f"{grp_7m_sg:,}",
     ">32,330"),
    ("— Perturbation stats: all cells (no single-guide filter) —", "", ""),
    ("  Unique gene targets",
     f"{n_tgt_7m_all:,}",
     "—"),
    ("  Median nuclei per perturbation",
     f"{med_7m_all:,}",
     "—"),
    ("  Pert × subclass pairs >50 cells",
     f"{sub_7m_all:,}",
     "—"),
    ("  Pert × group pairs >50 cells",
     f"{grp_7m_all:,}",
     "—"),
]

df_7m = pd.DataFrame(rows_7m, columns=["Statistic", "Data (full 7.7M obs)", "Paper claim"])
df_7m.set_index("Statistic", inplace=True)

print("=" * 70)
print("TABLE 1 — Full 7.7M obs")
print("=" * 70)
display(df_7m)

# ── Table 2: QC-filtered 6.35M obs ──────────────────────────────────────────
rows_filt = [
    ("Total nuclei (after QC filter)",
     f"{pp_cells:,}  ({pp_cells/total*100:.1f}% of raw)",
     "~6.35M (fig: 6M)"),
    ("Median UMI/nucleus — range across neuronal neighborhoods",
     f"{umi_min_filt:,} – {umi_max_filt:,}",
     "—"),
    ("% nuclei with guide detected",
     f"{guide_all_pct_filt:.2f}%",
     "—"),
    ("% nuclei with single guide",
     f"{single_all_pct_filt:.2f}%",
     "—"),
    ("— Perturbation stats: single-guide cells only —", "", ""),
    ("  Unique gene targets",
     f"{n_tgt_fi_sg:,}",
     "—"),
    ("  Median nuclei per perturbation",
     f"{med_fi_sg:,}",
     "—"),
    ("  Pert × subclass pairs >50 cells",
     f"{sub_fi_sg:,}",
     "—"),
    ("  Pert × group pairs >50 cells",
     f"{grp_fi_sg:,}",
     "—"),
    ("— Perturbation stats: all cells (no single-guide filter) —", "", ""),
    ("  Unique gene targets",
     f"{n_tgt_fi_all:,}",
     "—"),
    ("  Median nuclei per perturbation",
     f"{med_fi_all:,}",
     "—"),
    ("  Pert × subclass pairs >50 cells",
     f"{sub_fi_all:,}",
     "—"),
    ("  Pert × group pairs >50 cells",
     f"{grp_fi_all:,}",
     "—"),
]

df_filt = pd.DataFrame(rows_filt, columns=["Statistic", "Data (QC-filtered 6.35M obs)", "Paper claim"])
df_filt.set_index("Statistic", inplace=True)

print("\n" + "=" * 70)
print("TABLE 2 — QC-filtered 6.35M obs  (singlet & num_genes>=2000 & log_ambient_mse_norm>0.09)")
print("=" * 70)
display(df_filt)

TABLE 1 — Full 7.7M obs


,Data (full 7.7M obs),Paper claim
Statistic,,
Total nuclei,"7,720,300","7,720,300"
Cell classes (predicted_class),34,34
Cell subclasses (predicted_subclass),331,331
Neighborhoods,7 core (+4 compound overlap labels),7
Median UMI/nucleus — range across neuronal neighborhoods,"12,146 – 15,548","11,404 – 14,652"
% nuclei with guide detected,66.49%,>66.49%
% nuclei with single guide,50.01%,50%
— Perturbation stats: single-guide cells only —,,
Unique gene targets,"1,949","~2,000"



TABLE 2 — QC-filtered 6.35M obs  (singlet & num_genes>=2000 & log_ambient_mse_norm>0.09)


,Data (QC-filtered 6.35M obs),Paper claim
Statistic,,
Total nuclei (after QC filter),"6,354,986 (82.3% of raw)",~6.35M (fig: 6M)
Median UMI/nucleus — range across neuronal neighborhoods,"11,404 – 14,652",—
% nuclei with guide detected,67.99%,—
% nuclei with single guide,55.54%,—
— Perturbation stats: single-guide cells only —,,
Unique gene targets,"1,949",—
Median nuclei per perturbation,"1,637",—
Pert × subclass pairs >50 cells,"13,583",—
Pert × group pairs >50 cells,"29,539",—


In [ ]:
obs["predicted_group"].nunique()

31